# VoiceFlow: Speech-to-Text and Text-to-Speech Conversion System
### BCA Final Year Project - Google Colab Edition
This notebook runs 100% on Google Colab without external datasets or paid APIs.

In [ ]:
# Step 1: Install audio tools and Python libraries
!apt-get update -qq
!apt-get install -y -qq espeak espeak-ng ffmpeg
!pip install -q gradio SpeechRecognition gTTS pydub
print('All dependencies installed successfully!')

In [ ]:
# Step 2: Run the Complete Speech System
import os
import speech_recognition as sr
from gtts import gTTS
import gradio as gr

def speech_to_text(audio_path):
    if not audio_path:
        return 'Error: Please record or upload an audio file first.'
    recognizer = sr.Recognizer()
    temp_wav = 'colab_temp.wav'
    os.system(f"ffmpeg -y -i '{audio_path}' -ar 16000 -ac 1 '{temp_wav}' > /dev/null 2>&1")
    try:
        with sr.AudioFile(temp_wav) as source:
            audio = recognizer.record(source)
        return recognizer.recognize_google(audio)
    except sr.UnknownValueError:
        return 'Error: Speech was unintelligible. Please speak clearly.'
    except sr.RequestError:
        return 'Error: Speech recognition service unavailable. Check internet connection.'
    except Exception as e:
        return f'Error: {str(e)}'
    finally:
        if os.path.exists(temp_wav):
            os.remove(temp_wav)

def text_to_speech(text):
    if not text or not text.strip():
        return None
    out_path = 'synthesized_speech.mp3'
    tts = gTTS(text=text.strip(), lang='en', slow=False)
    tts.save(out_path)
    return out_path

def compute_wer(ref, hyp):
    ref_words = ref.strip().lower().split()
    hyp_words = hyp.strip().lower().split()
    r, h = len(ref_words), len(hyp_words)
    if r == 0:
        return 'Reference text cannot be empty.'
    dp = [[0]*(h+1) for _ in range(r+1)]
    for i in range(r+1): dp[i][0] = i
    for j in range(h+1): dp[0][j] = j
    for i in range(1, r+1):
        for j in range(1, h+1):
            if ref_words[i-1] == hyp_words[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = min(dp[i-1][j-1]+1, dp[i][j-1]+1, dp[i-1][j]+1)
    wer = round((dp[r][h] / r) * 100, 2)
    acc = round(max(0.0, 100.0 - wer), 2)
    i, j = r, h
    sub, ins, dlt, cor = 0, 0, 0, 0
    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref_words[i-1] == hyp_words[j-1]:
            cor += 1; i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            sub += 1; i -= 1; j -= 1
        elif j > 0 and dp[i][j] == dp[i][j-1] + 1:
            ins += 1; j -= 1
        else:
            dlt += 1; i -= 1
    return f"### Evaluation Results:\n- **Word Error Rate (WER):** {wer}%\n- **Recognition Accuracy:** {acc}%\n- **Substitutions (S):** {sub}\n- **Deletions (D):** {dlt}\n- **Insertions (I):** {ins}\n- **Correct Words (C):** {cor}\n- **Total Reference Words (N):** {r}"

with gr.Blocks(title='VoiceFlow - Speech System') as demo:
    gr.Markdown('# VoiceFlow: Speech-to-Text and Text-to-Speech System')
    gr.Markdown('**BCA Final Project** | Voice Processing and WER Evaluation')
    with gr.Tab('Speech-to-Text (STT)'):
        audio_in = gr.Audio(sources=['microphone', 'upload'], type='filepath', label='Record or Upload Audio')
        btn_stt = gr.Button('Transcribe Audio', variant='primary')
        text_out = gr.Textbox(label='Recognized Text', lines=4)
        btn_stt.click(speech_to_text, inputs=audio_in, outputs=text_out)
    with gr.Tab('Text-to-Speech (TTS)'):
        text_in = gr.Textbox(label='Enter Text to Speak', lines=3, placeholder='Type text here...')
        btn_tts = gr.Button('Generate Speech', variant="primary")
        audio_out = gr.Audio(label='Generated Speech (Play and Download)')
        btn_tts.click(text_to_speech, inputs=text_in, outputs=audio_out)
    with gr.Tab('AI/ML WER Evaluator'):
        ref_in = gr.Textbox(label='Reference Text (Ground Truth)', lines=2)
        hyp_in = gr.Textbox(label='Recognized Text (Hypothesis)', lines=2)
        btn_wer = gr.Button('Calculate WER and Accuracy', variant='primary')
        wer_out = gr.Markdown()
        btn_wer.click(compute_wer, inputs=[ref_in, hyp_in], outputs=wer_out)

demo.launch(share=True)
